# Session Data Export
Extracts orders, signals, and portfolio snapshots for a given session from MongoDB and writes them to an Excel file.

In [ ]:
# --- Parameters ---
MONGO_URL   = "mongodb://localhost:27017"
DATABASE    = "trading"
SESSION_ID  = "your-session-id-here"

OUTPUT_FILE = f"output/session_{SESSION_ID}.csv"

In [3]:
import os
import pandas as pd
from pymongo import MongoClient

client = MongoClient(MONGO_URL)
db = client[DATABASE]

raw_orders    = list(db["orders"].find({"session_id": SESSION_ID}))
raw_signals   = list(db["signals"].find({"session_id": SESSION_ID}))
raw_snapshots = list(db["portfolio_snapshots"].find({"session_id": SESSION_ID}))

print(f"orders: {len(raw_orders)}, signals: {len(raw_signals)}, snapshots: {len(raw_snapshots)}")

orders: 8, signals: 9, snapshots: 13337


In [4]:
def mongo_to_df(docs, sort_col="timestamp"):
    """Convert a list of Mongo documents to a DataFrame, dropping _id."""
    df = pd.DataFrame(docs)
    if df.empty:
        return df
    df = df.drop(columns=["_id"], errors="ignore")
    if sort_col in df.columns:
        df[sort_col] = pd.to_datetime(df[sort_col], utc=True, errors="coerce")
        df = df.sort_values(sort_col).reset_index(drop=True)
    return df


def flatten_metadata(df, meta_col="metadata"):
    """Expand a dict column into individual columns prefixed with 'meta_'."""
    if meta_col not in df.columns:
        return df
    expanded = df[meta_col].apply(
        lambda v: v if isinstance(v, dict) else {}
    ).apply(pd.Series)
    expanded.columns = [f"meta_{c}" for c in expanded.columns]
    df = df.drop(columns=[meta_col]).join(expanded)
    return df


df_orders    = mongo_to_df(raw_orders)
df_signals   = flatten_metadata(mongo_to_df(raw_signals))
df_snapshots = mongo_to_df(raw_snapshots)

print("orders shape:   ", df_orders.shape)
print("signals shape:  ", df_signals.shape)
print("snapshots shape:", df_snapshots.shape)

orders shape:    (8, 20)
signals shape:   (9, 13)
snapshots shape: (13337, 5)


In [11]:
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

exports = {
    "orders":               df_orders,
    "signals":              df_signals,
    "portfolio_snapshots":  df_snapshots,
}

for name, df in exports.items():
    path = OUTPUT_FILE.replace(".xlsx", f"_{name}.csv")
    df.to_csv(path, index=False)
    print(f"Exported {len(df):>6} rows → {path}")

Exported      8 rows → output/session_live_SPY_switching_2214_1716_1722_orders.csv
Exported      9 rows → output/session_live_SPY_switching_2214_1716_1722_signals.csv
Exported  13337 rows → output/session_live_SPY_switching_2214_1716_1722_portfolio_snapshots.csv
